# VisDrone üzərində YOLOv8 fine-tuning (kiçik obyektlər üçün)

Dron şəkillərindəki kiçik obyektləri (insan, maşın, motosiklet və s.) daha yaxşı tapan model öyrətmək.

Runtime → Change runtime type → **T4 GPU**.

1) GPU yoxla → 2) quraşdır → 3) Drive qoş → 4) təlim → 5) qiymətləndirmə → 6) `best.pt`



In [1]:
!nvidia-smi

Tue Oct 20 11:16:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 15.6 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from ultralytics import YOLO

PROJECT = "/content/drive/MyDrive/yolo_runs"
NAME = "visdrone_yolov8s_960"

model = YOLO("yolov8s.pt")
results = model.train(
    data="VisDrone.yaml",
    epochs=10,
    imgsz=960,
    batch=16,
    patience=10,
    workers=2,
    project=PROJECT,
    name=NAME,
    exist_ok=True,
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.183 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_

### Colab ortada bağlandısa (davam etdirmək üçün)
Aşağıdakı xananı işlət, təlim qaldığı yerdən davam edər.

In [5]:
# YALNIZ təlim yarımçıq qalıbsa işlət:
# from ultralytics import YOLO
# YOLO("/content/drive/MyDrive/yolo_runs/visdrone_yolov8s_960/weights/last.pt").train(resume=True)

In [6]:
best = YOLO(f"{PROJECT}/{NAME}/weights/best.pt")
metrics = best.val(data="VisDrone.yaml", imgsz=960)

print(f"mAP50    : {metrics.box.map50:.3f}")
print(f"mAP50-95 : {metrics.box.map:.3f}")
print("\nSinif üzrə mAP50-95:")
for idx, score in zip(metrics.box.ap_class_index, metrics.box.maps[metrics.box.ap_class_index]):
    print(f"  {best.names[int(idx)]:<18} {score:.3f}")

Ultralytics 8.4.183 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,129,454 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2962.4±1006.2 MB/s, size: 153.1 KB)
val: Scanning /content/datasets/VisDrone/labels/val.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 191.5Mit/s 0.0s
WARNING ⚠️ Dataset images contain up to 317 objects (val=317), but max_det=300. This mismatch can cap recall and produce invalid validation metrics. Raising it may increase validation cost but cannot increase model or export capacity, which may cap recall. Setting max_det=317 to match the observed maximum.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 2.0it/s 17.3s0.4s
                   all        548      38759      0.557      0.431      0.435       0.26
            pedestrian        520       8844      0.619      0.476      0.504 

In [7]:
from google.colab import files
files.download(f"{PROJECT}/{NAME}/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import os
print(os.path.ismount("/content/drive"))
!ls -lh /content/drive/MyDrive/yolo_runs/visdrone_yolov8s_960/weights

True
total 43M
-rw------- 1 root root 22M Oct 20 12:01 best.pt
-rw------- 1 root root 22M Oct 20 12:01 last.pt
